# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step example for loading and exploring the [FAIR² Croissant](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api/notebooks/) library.

### Dataset Source
The dataset schema is accessed via a Croissant JSON-LD URL and supports automated data loading and field introspection.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata and inspect high-level information about the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema JSON-LD)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Display dataset name and description
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and their fields. All elements are referenced by their `@id` for precise selection and extraction later.

In [ ]:
record_sets = dataset.metadata.record_sets  # List of RecordSet objects

for rs in record_sets:
    print("Record Set Name:", getattr(rs, 'name', None))
    print("  @id:     ", rs.id)
    print("  Description:", getattr(rs, 'description', ''))
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {getattr(field, 'name', '')} (@id: {field.id}, dataType: {getattr(field, 'data_type', '')})")
    print()

## 3. Data Extraction
Load data from one or more record sets as DataFrames for downstream analysis.
All record sets and fields are referenced by their `@id` as per Croissant best practices.

In [ ]:
# List all record set @ids for extraction
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Loading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    # Create DataFrame if records exist
    if records and isinstance(records[0], dict):
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  Fields: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  No records or unable to convert to DataFrame.")

# Choose the first available record set for further analysis
main_record_set_id = next(iter(dataframes), None)  # Take the first DataFrame found
if main_record_set_id is not None:
    print(f"\nSelected main RecordSet for analysis: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We will select a numeric field (by its `@id`) to demonstrate filtering and normalization, and group by a categorical field where available.

In [ ]:
# If main_record_set_id was found and DataFrame is present
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print(f"Fields available for analysis: {df.columns.tolist()}")
    # Try to automatically pick a numeric field
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    # If no numeric columns, try to coerce one
    if not numeric_fields and len(df.columns) > 0:
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except:
                continue
        numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Use the first numeric field found
        print(f"Numeric field chosen for EDA: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    else:
        print("No suitable numeric field found for EDA.")

    # Attempt to find a grouping field (categorical/non-numeric w/ low unique count)
    grouping_field = None
    for col in df.columns:
        if df[col].dtype == object and df[col].nunique() < min(10, len(df)//5):
            grouping_field = col
            break

    if grouping_field and numeric_fields:
        grouped_df = filtered_df.groupby(grouping_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {grouping_field} (mean {numeric_field}):")
        print(grouped_df)
    else:
        print("No suitable grouping field detected.")
else:
    print("No main record set loaded for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and comparisons by group if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouped data available, show comparison
    if grouping_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=grouping_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {grouping_field}")
        plt.show()

## 6. Conclusion
 - Dataset accessed and loaded using Croissant schema and `mlcroissant`.
 - Record sets and fields are referenced throughout by their `@id`, making analyses robust and reproducible.
 - Numeric fields were filtered, normalized, and visualized; group-wise statistics provided as available.
 - The data is now ready for specialized biological, statistical, or machine learning analyses.

_For additional details and API usage examples, refer to the [`mlcroissant` documentation](https://mlcommons.github.io/croissant/api/notebooks/)._